## Topic: RunnableLambda

### Agenda 
- 1. Introduction of RunnableLambda

- 2. The Rule:

- 3. Practical Example of RunnableLambda

- 4. Summary RunnableLambda

### 1. Introduction of RunnableLambda
- Definition:
    - RunnableLambda is a runnable primitive that allows you to apply custom python functions within an al Pipeline.

    - RunnableLambda is a LangChain Runnable that wraps any Python function (regular function, lambda, or async function) so it can be used inside LCEL chains with the | pipe operator.


- Benefit:
    - It acts as a middleware between different AI components, enabling preprocessing, transformation, API calls, filtering, and post-processing in a LangChain workflow.

In [ ]:
"""    
- The Core Idea:
┌─────────────────────────────────────────────────────────────┐
│                    RunnableLambda                           │
│                                                             │
│  Your Python function:                                      │
│    def my_func(x): return x.upper()                         │
│                                                             │
│  Problem: Can't use it in a chain with |                    │
│    chain = prompt | llm | my_func  ←    TypeError!          │
│                                                             │
│  Solution: Wrap it in RunnableLambda                        │
│    from langchain_core.runnables import RunnableLambda      │
│    chain = prompt | llm | RunnableLambda(my_func)  ←        │
│                                                             │
│  Now your function is a full Runnable with:                 │
│    .invoke()  .stream()  .batch()  .ainvoke()  .astream()   │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│              WHY RunnableLambda?                            │
├─────────────────────────────────────────────────────────────┤
│ 1. -  COMPOSABILITY                                         │
│    Plug any Python function into a chain with |             │
│                                                             │
│ 2. -  TRACING                                               │
│    Your function shows up in LangSmith traces               │
│                                                             │
│ 3. -  STREAMING & BATCHING                                  │
│    Your function inherits .stream() and .batch()            │
│                                                             │
│ 4. -  TRANSFORMATION                                        │
│    Reshape data between chain steps (dict → str, etc.)      │
│                                                             │
│ 5. -  CUSTOM LOGIC                                          │
│    Add business logic, filtering, formatting, validation    │
└─────────────────────────────────────────────────────────────┘

"""

### RunnableLambda VS RunnablePassthrough VS RunnableParallel

In [ ]:
   ## RunnableLambda VS RunnablePassthrough VS RunnableParallel

""" 
┌──────────────────────┬──────────────────┬──────────────────┬──────────────────┐
│ Feature              │ RunnableLambda   │ RunnablePass     │ RunnableParallel │
│                      │                  │ through          │                  │
├──────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Purpose              │ Transform data   │ Pass data        │ Run multiple     │
│                      │ with a function  │ unchanged        │ chains at once   │
├──────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Input → Output       │ Any → Any        │ Same → Same      │ Any → Dict       │
│                      │ (transforms)     │ (no change)      │ (merges)         │
├──────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Function required?   │    Yes           │    No            │    No (chains)   │
├──────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Changes data?        │    Yes           │    No            │    Yes           │
├──────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Use case             │ Clean, reshape,  │ Keep original    │ Multi-analysis,  │
│                      │ convert, enrich  │ input in RAG     │ fan-out          │
├──────────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Example              │ lambda x:        │ PassThrough()    │ {a: chain1,      │
│                      │   x.upper()      │                  │  b: chain2}      │
└──────────────────────┴──────────────────┴──────────────────┴──────────────────┘

- Key Note:

    - Sequential Runnable → fixed order

    - RunnableParallel → independent branches

    - RunnablePassthrough → keep input unchanged

    - RunnableLambda converts a normal Python function into a LangChain Runnable.


"""

### 2. The Rule:
- RunnableLambda takes one input and returns one output. The types can be anything

In [ ]:
"""   - The Rule: 

┌─────────────────────────────────────────────────────────────┐
│           RunnableLambda INPUT/OUTPUT TYPES                 │
│                                                             │
│  Input Type    →  Output Type    →  Use Case                │
│  ──────────       ──────────        ────────                │
│  str          →  str             →  Text transformation     │
│  str          →  int             →  Word count              │
│  str          →  list[str]       →  Sentence splitting      │
│  dict         →  dict            →  Field enrichment        │
│  dict         →  str             →  Field extraction        │
│  AIMessage    →  str             →  Content extraction      │
│  list[Doc]    →  str             →  Document formatting     │
│  Any          →  Any             →  Anything!               │
│                                                             │
│  -  IMPORTANT: The function receives exactly ONE argument   │
│     (the output of the previous chain step).                │
└─────────────────────────────────────────────────────────────┘

"""

### 3. Practical Example of RunnableLambda


In [1]:
# Example 3.1: RunnableLambda
# Purpose: How to work RunnableLambda

from langchain_core.runnables import RunnableLambda


# python function
def word_counter(text):
    return len(text.split())


runnabe_word_count = RunnableLambda(word_counter)

response = runnabe_word_count.invoke("Hello! My name is KzRaihan")

print(response)

5


#### Example 3.2:
- Idea:
    - step1: 
        - prompt1:  Generate a joke base on {topic} 

    - step2: RunnableSequence
        - input: prompt1
        - process: RunnableSequence
        - output: joke on {topic}


    - step3: RunnableParallel 
        - input: 
        - process: RunnableParallel
                        -> prompt1 -> RunnablePassThrough ---> parser
                        -> Prompt2 -> RunnableLambda -> parser
        
        
        - output: 
            - Generate a joke about the topic.
            - count total number of word in Generated joke.


In [ ]:
# Example 3.2: convert python basic function to Runnable
# Purpose: Generate a joke base on {topic} and count the number of work in the response


from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassThrough, RunnableLambda

from dotenv import load_dotenv
load_dotenv()

# define a function 
def count_counter(text):
    return len(text.split())

# Define prompt1 -> write a joke on topic
prompt1 = PromptTemplate(
    template = "write a joke about {topic}",
    input_variables=["topic"]
)


# define model
model = ChatOpenAI()

# Define parser
parser = StrOutputParser()

# Create a  chain using  (LCEL)
joke_gen_chain = RunnableSequence(prompt1 | model | parser)

# parallel chain for -> 1. original text, 2. work count of the joke
parallel_chain = RunnableParallel({
    'joke': RunnablePassThrough(),
    "word_count": RunnableLambda(count_counter)

})

# another way
# parallel_chain = RunnableParallel({
#     'joke': RunnablePassThrough(),
#     "word_count": RunnableLambda(lambda x:len(x.split()))

# })



# final chain for connection between joke_gen_chain and parallel_chain
final_chain = RunnableSequence(joke_gen_chain | parallel_chain)


# response
response = chain.invoke(
    {
        "topic": "GenAI"
    }
)

print(f"Response: \n {response}")

# formatting
final_response = """ {} \n Word count - {}""".format(response['joke'], response['word_count'])

print(final_response)

### 4. Complete Summary RunnableLambda

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                      RunnableLambda                              │
│                                                                  │
│  WHAT:  Wraps any Python function as a chainable Runnable        │
│  WHY:   Plug custom logic into LCEL pipelines with |             │
│  HOW:   RunnableLambda(my_function) or RunnableLambda(lambda x)  │
│                                                                  │
│  SYNTAX:                                                         │
│    chain = prompt | llm | parser | RunnableLambda(my_func)       │
│                                                                  │
│  INPUT:   Exactly 1 argument (output of previous step)           │
│  OUTPUT:  Any type (must match next step's expected input)       │
│                                                                  │
│  FEATURES:                                                       │
│    ├── .invoke()   → Single execution                            │
│    ├── .stream()   → Yields result (single yield)                │
│    ├── .batch()    → Maps function over inputs                   │
│    ├── .ainvoke()  → Async execution                             │
│    ├── Config access → func(input, config)                       │
│    └── Async support → async def func(input)                     │
│                                                                  │
│  COMMON USE CASES:                                               │
│    ├── Data reshaping (dict → str, list → str)                   │
│    ├── Text cleaning (strip markdown, remove HTML)               │
│    ├── Field extraction (x["key"])                               │
│    ├── Post-processing (validate, normalize, format)             │
│    ├── External API calls (enrich with live data)                │
│    ├── Merging parallel outputs (dict → formatted string)        │
│    └── Conditional logic (if/else formatting)                    │
│                                                                  │
│  VS OTHER RUNNABLES:                                             │
│    RunnableLambda      → Transform data with a function          │
│    RunnablePassthrough → Pass data unchanged                     │
│    RunnableParallel    → Run multiple chains simultaneously      │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "If you need custom Python logic in a chain, wrap it in         │
│   RunnableLambda. Keep functions pure, typed, and tested."       │
└──────────────────────────────────────────────────────────────────┘


"""